## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

Enter your Anthropic API key:  ········
Enter your Tavily API key:  ········


## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [19]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

 THe 3 states, AgentState (global), SupervisorState (coordination), ResearcherState (per worker).
 
 1. overall AgentState - highest level overview of messages, the research brief, accumulated notes, and (eventually) the final report.
 2. SupervisorState that is informmed - oordinates the work—tracks the supervisor’s messages, how many research iterations have run, and which sub-tasks to delegate.
 3. ResearchState - each researcher’s workspace comprising messages, tool calls, and findings.

Separating states allow for:
 1. Separation of concers
 2. Parallelism and scaling of each research agent's work
 3. Smaller context window as each agent prompts what it needs only
 4. Easier to determine part that changes and rerun that section again if necessary.

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [20]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

### Advantages:

Modularity and Reuse across different notebooks and projects.

Smaller Notebook Files.

Faster Debugging and Testing as each module or file could be separately tested

### Disadvantages:

Setup Overhead.

Harder Traceability - it’s less obvious how they work unless you open the source.

Dependency Management – Imported modules may rely on external packages.

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [21]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [22]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [23]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [24]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [25]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [26]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [27]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [28]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [29]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [30]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [31]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [32]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [33]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [34]:
# Improved research request that triggers clarification
research_request_improved = """
I want to research how people are using AI/ChatGPT. I'm interested in understanding usage patterns, trends, and insights about AI adoption.

What specific aspects would you like me to focus on for this research?
"""

print("✓ Improved research request ready")
print("This request will trigger clarification questions from the system")


✓ Improved research request ready
This request will trigger clarification questions from the system


In [35]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


In [36]:
# Execute research with improved request that triggers clarification
async def run_improved_research():
    """Run the research workflow with improved request that triggers clarification."""
    print("Starting improved research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request_improved}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Improved research workflow completed!")
    print("="*60)

# Run the improved research
await run_improved_research()


Starting improved research workflow...


Node: clarify_with_user

To provide you with the most relevant research on AI/ChatGPT usage, I'd like to clarify a few key aspects:

**Scope & Focus:**
- Are you looking for consumer/general public usage or business/enterprise adoption (or both)?
- Do you want to focus specifically on ChatGPT, or include other AI tools like Claude, Gemini, etc.?

**Geographic Coverage:**
- Any specific regions/countries of interest, or should this be global?

**Time Frame:**
- What time period should I cover? (e.g., last 6 months, year-over-year trends, since ChatGPT's launch)

**Key Areas of Interest:**
- Usage statistics and demographics
- Industry/sector adoption patterns  
- Popular use cases and applications
- Barriers to adoption
- Future trends and predictions

Please let me know which of these areas are most important for your research, and if there are any specific angles or insights you're particularly interested in.

Improved research workflow complet

## Activity #1: Improved Clarification Configuration

**Changes Made:**
1. **Modified Research Request**: Changed from PDF analysis to open-ended AI usage research
2. **Triggered Clarification**: The new request asks "What specific aspects would you like me to focus on?" 
3. **Added Improved Execution**: Created new cells (38-39) with better research request and execution

**Why These Changes:**
- **Original Issue**: The PDF-based request bypassed clarification and led to "technical limitations" messages
- **Solution**: Open-ended request triggers the clarification phase, allowing you to specify research focus
- **Expected Result**: System will ask clarifying questions, then conduct focused web searches with real sources

**Configuration Used:**
- `allow_clarification: True` - Enables the clarification phase
- `max_concurrent_research_units: 3` - Moderate parallelism for better coverage
- `max_researcher_iterations: 4` - More delegation rounds for thorough research
- `max_react_tool_calls: 5` - More searches per researcher for comprehensive results
- `search_api: "tavily"` - Web search for real sources


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [37]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...May last a few minutes\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with analyzing the ChatGPT usage research paper. I understand you want me to provide insights on: (1) main findings about how people use AI/ChatGPT, (2) most common use cases, and (3) trends and patterns from the data. The PDF contains a comprehensive NBER working paper with detailed usage statistics, classifications, and demographic breakdowns from ChatGPT's launch through July 2025. I will now analyze this document and provide the requested insights based on the research findings.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER Working Paper "How People Use ChatGPT" (Working Paper 34255, September 2025) by Aaron Chatterji et al. Please provide detailed insights on: (1) the main findings about how people are using ChatGPT/AI based on their analysis of ChatGPT usage data from November 2022 through July 2025, including adoption rates, demog

# Comprehensive Analysis of the NBER Working Paper "How People Use ChatGPT"

## Executive Summary

The NBER Working Paper 34255 "How People Use ChatGPT" by Aaron Chatterji et al. represents the most comprehensive privacy-preserving analysis of real ChatGPT usage patterns ever conducted [1]. Published in September 2025, this groundbreaking study analyzes approximately 1.5 million de-identified conversations from ChatGPT's consumer platform, revealing unprecedented insights into how people actually use large language models in practice [2]. The research demonstrates ChatGPT's extraordinary global adoption—reaching 10% of the world's adult population by July 2025 with 700 million users sending 18 billion messages weekly—while uncovering significant shifts in usage patterns, demographic adoption trends, and the primary ways people derive value from AI assistance [3].

## Global Adoption and Unprecedented Growth

ChatGPT's adoption trajectory has been historically unprecedented. By July 2025, the platform reached 750+ million weekly active users, representing nearly 10% of the world's population [4]. The speed of this diffusion outpaces any previous consumer technology—ChatGPT reached 100 million weekly active users after just one year and 350 million after two years [1].

The scale of usage is remarkable: as of June 2025, users were sending more than 2.6 billion messages per day, equivalent to over 30,000 messages per second [4]. Message volume experienced explosive growth, increasing 5.8 times in the past year alone, with the platform reaching 1 billion daily messages in December 2024—less than two years after launch, compared to Google's eight years to reach 1 billion daily searches [4].

User engagement patterns reveal deepening usage intensity over time. Weekly active users doubled every 7-8 months since reaching 100 million in November 2023, with all user cohorts showing similar patterns of flat usage through 2024 followed by substantial increases in early 2025, suggesting significant platform improvements and enhanced user-friendliness [4].

## Dramatic Shift from Work to Non-Work Usage

One of the study's most significant findings is the pronounced shift from work-related to non-work usage. Between June 2024 and June 2025, non-work messages grew from 53% to over 70% of all ChatGPT usage [1][2][3]. This shift occurred within existing user cohorts rather than due to changing user composition, indicating that people are finding increasingly diverse applications for ChatGPT beyond workplace productivity.

The data reveals that approximately 30% of consumer usage remains work-related while 70% is non-work related, with both categories continuing to grow in absolute terms [2]. This pattern suggests that ChatGPT's consumer surplus extends significantly beyond workplace applications, aligning with research by Collis and Brynjolfsson (2025) estimating at least $97 billion in US consumer surplus from generative AI in 2024 alone [1].

Work usage patterns show distinct characteristics: work-related messages are more common among educated users in highly-paid professional occupations, with higher-skill, knowledge-intensive professionals being more likely to use ChatGPT for work purposes [1][5]. The research found that 81% of work-related messages involve obtaining/interpreting information and decision-making/problem-solving activities [1].

## Primary Use Case Categories and Taxonomy

The researchers developed a comprehensive taxonomy revealing that nearly 80% of all ChatGPT usage falls into three primary categories [1][2][3]:

### Practical Guidance (29% of Usage)
This category represents the most common use case, encompassing tutoring and teaching, how-to advice across various topics, and creative ideation. Notably, about 10% of all messages are requests for tutoring or teaching, highlighting education as a key application for ChatGPT [1]. This category demonstrates ChatGPT's role as a versatile advisor and learning companion.

### Seeking Information (24% of Usage)  
This category includes searching for facts about people, current events, products, and recipes, appearing to serve as a close substitute for traditional web search. However, unlike search engines, ChatGPT provides more conversational, contextual responses that can be tailored to specific user needs and follow-up questions.

### Writing (24% of Usage)
Writing encompasses automated production of emails, documents, and other communications, as well as editing, critiquing, summarizing, and translating user-provided text. This category dominates work-related tasks, accounting for 40% of work-related messages on average in June 2025 [1][3]. Importantly, about two-thirds of all Writing messages ask ChatGPT to modify existing user text rather than creating entirely new content from scratch [1].

### Other Notable Categories
Computer programming represents only 4.2% of all messages—much lower than many assumptions about AI usage—while companionship and social-emotional uses account for just 1.9% of conversations [1][7]. This data challenges common perceptions about the primary drivers of AI adoption.

## User Intent Classification System

The researchers introduced an innovative classification system based on user intent, revealing that 49% of messages are "Asking" (seeking information or advice for decision-making), 40% are "Doing" (requesting task performance or output creation), and 11% are "Expressing" (social or emotional content) [1][2].

The prevalence of "Asking" messages—which represent the fastest-growing and highest-rated category—demonstrates that people value ChatGPT most as an advisor rather than simply as a task completion tool [2][4]. In work contexts, "Doing" messages comprise 56% of usage, with nearly three-quarters being writing-related tasks [1]. This pattern highlights ChatGPT's unique ability to generate tailored, actionable outputs compared to traditional search engines.

## Demographic Evolution and Narrowing Gaps

The study reveals dramatic demographic shifts in ChatGPT adoption. Early adopters were predominantly male (80%), but the gender gap has narrowed remarkably—feminine names increased from 37% in January 2024 to 52% by July 2025, suggesting the gender gap may have closed completely [2][4].

Age patterns show that nearly half of all adult messages come from users under 26, though age gaps are also narrowing over time [1][3]. This concentration among younger users may partly explain the shift toward non-work usage, as younger users may have different application priorities.

The demographic evolution reflects ChatGPT's transformation from an early-adopter technology to a mainstream consumer platform serving diverse populations with varying needs and use cases.

## Geographic and Economic Adoption Patterns

Global adoption patterns reveal accelerating growth, particularly in low- and middle-income countries where growth rates are approximately 4 times higher than in the highest-income countries as of May 2025 [2]. Middle-income countries showed 5-6 times growth in usage compared to 3 times growth in the richest countries [4].

This pattern breaks conventional assumptions about technology adoption, with countries like Brazil, South Korea, and the US now showing similar usage rates despite vastly different GDP per capita levels [4]. The faster adoption in lower-income countries challenges the notion of AI being exclusively a "rich world" technology [3].

These geographic patterns suggest that language models may be particularly valuable in contexts where access to information, education, and professional services has traditionally been limited, potentially serving as an equalizing force in global access to knowledge and assistance.

## Educational and Professional Usage Patterns

The research reveals significant variations in usage patterns across education levels and professional occupations. Work usage is substantially more common among educated users in highly-paid professional occupations, with knowledge-intensive professionals showing the highest propensity for work-related ChatGPT use [1][5][8].

For professional users, writing tasks dominate work-related applications, highlighting how ChatGPT complements knowledge work by automating routine communication and document creation tasks. The finding that 56% of work-related messages are "Doing" tasks—with nearly three-quarters being writing-related—demonstrates ChatGPT's particular value in professional contexts requiring significant written communication [1].

The educational implications are substantial, with tutoring and teaching requests comprising 10% of all messages, suggesting widespread adoption for learning and skill development across diverse subjects and contexts [1].

## User Satisfaction and Quality Improvements

User satisfaction metrics indicate substantial platform improvements over time. "Good" interactions are now 4 times more common than negative ones, with positive interactions outnumbering negative ones by approximately 4:1 [3][7]. The "Asking" category yields the highest-rated outcomes, reinforcing the value users place on ChatGPT's advisory capabilities [3].

The improvement in user satisfaction correlates with increased usage intensity across all user cohorts, particularly the substantial increases observed in early 2025, suggesting significant platform enhancements that made the technology more useful and user-friendly [4].

## Economic Value Creation and Decision Support

The study concludes that ChatGPT's primary economic value comes through decision support, which proves especially important in knowledge-intensive jobs where better decision-making directly increases productivity [1][2]. Unlike traditional productivity tools that automate specific tasks, ChatGPT enhances human judgment and decision-making capabilities across diverse contexts.

The economic impact extends beyond traditional workplace productivity measures. ChatGPT creates value through both increased workplace efficiency and personal benefits that traditional GDP measures may not fully capture [2][4]. This suggests that the technology's true economic contribution may be significantly underestimated by conventional economic metrics.

The emphasis on decision support rather than task replacement aligns with patterns showing that most writing requests involve editing and improving existing content rather than generating entirely new material, indicating ChatGPT serves more as an intelligent collaborator than a replacement for human capabilities [1].

## Comparison to Traditional Technologies and Unique Value Proposition

ChatGPT's usage patterns reveal fundamental differences from traditional technologies. Unlike search engines that provide links to information sources, ChatGPT generates tailored, actionable outputs that align with common workplace activities across various job functions [7]. This capability to produce customized content and advice represents a qualitative shift in how people interact with information technology.

The writing-dominated work usage highlights chatbots' unique ability to generate digital outputs compared to traditional search engines [1][3]. This generative capability, combined with conversational interaction patterns, positions ChatGPT as a fundamentally different type of tool that bridges information retrieval, content creation, and advisory services.

The unprecedented adoption speed—reaching 1 billion daily messages in under two years versus Google's eight years to reach 1 billion daily searches—suggests that generative AI fills previously unmet needs in how people seek assistance, create content, and make decisions [4].

### Sources

[1] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255
[2] How people are using ChatGPT | OpenAI: https://openai.com/index/how-people-are-using-chatgpt/
[3] ChatGPT: 700M users, 73% non-work use, 80% practical guidance: https://www.linkedin.com/posts/varundeep-kaur_how-chatgpt-is-being-used-activity-7373921179839094784-kX0c
[4] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt
[5] How People Actually Use ChatGPT — What 1.5M ...: https://medium.com/@adnanmasood/how-people-actually-use-chatgpt-what-1-5m-conversations-tell-us-about-the-next-decade-of-software-ea603212b458
[6] How People Actually Use ChatGPT — What 1.5M Conversations ...: https://medium.com/@adnanmasood/how-people-actually-use-chatgpt-what-1-5m-conversations-tell-us-about-the-next-decade-of-software-ea603212b458
[7] How People Really Use ChatGPT: Findings from NBER Research: https://techmaniacs.com/2025/09/15/how-people-really-use-chatgpt-findings-from-nber-research/
[8] OpenAI releases research on ChatGPT usage worldwide - LinkedIn: https://www.linkedin.com/posts/aaron-ronnie-chatterji_this-morning-the-openai-economic-research-activity-7373377911476649986-_n_s


Research workflow completed!


##  New configuration with increasing parallelism (max_concurrent_research_units: 5)

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

In [38]:
# Experiment 1: Increased Parallelism
config_parallelism = {
    "configurable": {
        # Copy all settings from original config
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior - CHANGED FOR EXPERIMENT
        "allow_clarification": True,
        "max_concurrent_research_units": 5,  # CHANGED: was 1, now 5
        "max_researcher_iterations": 2,      # Keep same
        "max_react_tool_calls": 3,           # Keep same
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Parallelism Experiment Configuration Ready")
print(f"  - Max Concurrent Researchers: 5 (was 1)")

✓ Parallelism Experiment Configuration Ready
  - Max Concurrent Researchers: 5 (was 1)


In [42]:
# Run parallelism experiment
import time

async def run_parallelism_experiment():
    """Run the research workflow with increased parallelism."""
    print("Starting PARALLELISM experiment...\n")
    start_time = time.time()
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request_improved}]},
        config_parallelism,  # Use experiment config
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    # Timing happens AFTER the entire workflow completes
    end_time = time.time()
    print(f"\n⏱️ PARALLELISM EXPERIMENT COMPLETED")
    print(f"⏱️ Total execution time: {end_time - start_time:.1f} seconds")
    print("="*60)

# Run the parallelism experiment
await run_parallelism_experiment()

Starting PARALLELISM experiment...


Node: clarify_with_user

To provide you with the most relevant research on AI/ChatGPT usage, I'd like to clarify a few key aspects:

**Scope & Focus:**
- Are you interested in general consumer usage, business/enterprise adoption, or both?
- Do you want to focus specifically on ChatGPT, or include other AI tools (Claude, Gemini, etc.)?

**Demographics & Geography:**
- Any specific demographics (age groups, professions, industries) you'd like me to focus on?
- Are you looking for global trends or specific regions/countries?

**Time Frame:**
- What time period should I cover? (e.g., last 6 months, year-over-year trends, since ChatGPT launch)

**Specific Metrics:**
- Are you most interested in adoption rates, frequency of use, specific use cases, or all of the above?
- Any particular insights you're hoping to uncover (e.g., productivity impact, learning applications, creative uses)?

Please let me know your preferences for these areas so I can tailor th

## The qustion was vague and clarifying questions were asked 

Passing clarification and resent query 

In [43]:
# Continue the parallelism experiment with clarification response
async def run_parallelism_experiment_with_clarification():
    """Run the research workflow with increased parallelism, including clarification response."""
    print("Starting PARALLELISM experiment with clarification response...\n")
    start_time = time.time()
    
    # Answer the clarifying questions that were asked
    clarification_response = """
Based on your questions, here are my preferences:

**Scope & Focus:** Both consumer and business usage, focusing specifically on ChatGPT
**Demographics & Geography:** Global trends, all demographics  
**Time Frame:** Last 6 months and year-over-year trends
**Specific Metrics:** All of the above - adoption rates, frequency, use cases, productivity impact, learning applications, and creative uses

Please proceed with comprehensive research covering all these areas.
"""
    
    # Run the workflow with the complete conversation history
    async for event in graph.astream(
        {"messages": [
            {"role": "user", "content": research_request_improved},
            {"role": "assistant", "content": "To provide you with the most relevant research on AI/ChatGPT usage, I'd like to clarify a few key aspects:\n\n**Scope & Focus:**\n- Are you interested in general consumer usage, business/enterprise adoption, or both?\n- Do you want to focus specifically on ChatGPT, or include other AI tools (Claude, Gemini, etc.)?\n\n**Demographics & Geography:**\n- Any specific demographics (age groups, professions, industries) you'd like me to focus on?\n- Are you looking for global trends or specific regions/countries?\n\n**Time Frame:**\n- What time period should I cover? (e.g., last 6 months, year-over-year trends, since ChatGPT launch)\n\n**Specific Metrics:**\n- Are you most interested in adoption rates, frequency of use, specific use cases, or all of the above?\n- Any particular insights you're hoping to uncover (e.g., productivity impact, learning applications, creative uses)?\n\nPlease let me know your preferences for these areas so I can tailor the research to your specific needs."},
            {"role": "user", "content": clarification_response}
        ]},
        config_parallelism,  # Use the existing parallelism config
        stream_mode="updates"
    ):
        # Use the same display logic as the existing function
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    # Timing happens AFTER the entire workflow completes
    end_time = time.time()
    print(f"\n⏱️ PARALLELISM EXPERIMENT WITH CLARIFICATION COMPLETED")
    print(f"⏱️ Total execution time: {end_time - start_time:.1f} seconds")
    print("="*60)

# Run the parallelism experiment with clarification
await run_parallelism_experiment_with_clarification()

Starting PARALLELISM experiment with clarification response...


Node: clarify_with_user

Perfect! I have all the information needed to conduct comprehensive research on ChatGPT usage patterns and trends. I'll now begin researching:

- **Consumer and business usage** of ChatGPT specifically
- **Global trends** across all demographics
- **Recent data** from the last 6 months plus year-over-year comparisons
- **Comprehensive metrics** including adoption rates, usage frequency, specific use cases, productivity impact, learning applications, and creative uses

I'll compile a detailed report covering all these aspects of ChatGPT adoption and usage patterns.

Node: write_research_brief

Research Brief Generated:
I need comprehensive research on ChatGPT usage patterns, trends, and insights covering both consumer and business adoption globally across all demographics. Specifically, I want to understand: (1) adoption rates and frequency of use in both consumer and enterprise contexts with year-


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Analysis of ChatGPT Usage Patterns, Trends, and Global Adoption

## Executive Summary

ChatGPT has achieved unprecedented global adoption, reaching 700-800 million weekly active users by September 2025, processing over 2.5 billion daily prompts globally [1][2]. The platform now represents the most rapidly adopted technology in human history, with over 80% of Fortune 500 companies implementing it within nine months of launch [3][4]. This comprehensive analysis reveals dramatic shifts in both consumer and enterprise usage patterns, with significant productivity gains, evolving demographic trends, and transformative business impacts across all sectors.

## Adoption Rates and Growth Trajectory

### Record-Breaking User Growth

ChatGPT's growth trajectory has shattered all previous technology adoption records. The platform achieved 1 million users in just 5 days after its November 2022 launch, reached 100 million monthly active users by January 2023 in just two months (compared to Facebook and Twitter taking 4-5 years to reach the same milestone), and expanded to 400 million weekly active users by February 2025 [1][2][5]. By July 2025, the platform processed 18 billion messages weekly from 700 million users, representing approximately 10% of the global adult population [6].

OpenAI projects ChatGPT will hit 1 billion users by the end of 2025 [2]. The platform currently receives approximately 4.61 billion monthly visits and maintains exceptional user loyalty, with 89% of paying subscribers staying enrolled for at least one quarter [1]. This retention rate demonstrates not just initial adoption but sustained value perception among users.

### Enterprise Penetration and Business Adoption

Enterprise adoption has been equally remarkable. Over 80% of Fortune 500 companies adopted ChatGPT within nine months of launch, with 92% of Fortune 100 companies now using the platform [1][3][4]. Currently, 49% of all companies use ChatGPT, with 93% of these organizations planning to expand their adoption [4]. Enterprise customers across Enterprise, Team, and Education offerings total 1.5 million subscribers [3].

Major brands leveraging ChatGPT for business operations include PwC, Canva, Zapier, and Klarna, spanning diverse industries from professional services to e-commerce [3]. This widespread corporate adoption indicates that ChatGPT has moved beyond experimental use to become integral business infrastructure.

### Financial Performance and Market Position

OpenAI's financial metrics reflect this explosive adoption. The company generated $3.7 billion in revenue in 2024, representing a 285% increase from the previous year, with ChatGPT accounting for 75% of OpenAI's total revenue [5]. The company achieved $10 billion annual recurring revenue (ARR) by June 2025 and projects $29.4 billion revenue by 2026 [1][3]. OpenAI's valuation has grown from $20 billion in December 2022 to between $157-500 billion by 2025 [1][5].

ChatGPT maintains market dominance with 82.7% market share, significantly ahead of competitors like Perplexity (8.2%) and Microsoft Copilot (4.5%) [4]. The platform holds 62.5% of the paid AI tool market and 79.76% of the chatbot market [1][3]. With 10-20 million paying subscribers across Plus, Team, and Pro tiers, ChatGPT generates $2.7 billion annually from paid subscriptions [2].

## Use Cases and Application Analysis

### Dominant Usage Categories

OpenAI's comprehensive analysis of usage patterns from November 2022 to July 2025 reveals that nearly 80% of all ChatGPT usage falls into three primary categories: "Practical Guidance" (customized advice and tutoring), "Seeking Information" (factual queries similar to web search), and "Writing" (creating and editing text) [6]. This distribution highlights ChatGPT's role as both an information resource and a productivity tool.

A significant trend has emerged in the work-versus-personal usage split. Non-work usage has grown dramatically, rising from 53% to over 70% of all messages between June 2024 and June 2025. This growth occurred both from new users entering the platform and changing behavior patterns within existing user cohorts [6].

### Writing and Content Creation Applications

Writing dominates work-related tasks, comprising 40% of all work messages. This prominence highlights chatbots' unique ability to generate digital outputs compared to traditional search engines [6]. Notably, two-thirds of writing tasks involve editing existing text rather than creating entirely new content, suggesting users leverage ChatGPT more for refinement than initial creation [6].

Primary content creation applications include media creation (28%) and information search (24%). Media creation breaks down into email writing (9%), essays (8%), creative writing (7%), and image generation (9%) [3]. Marketing agencies report a 40% reduction in content creation time due to ChatGPT's improved writing capabilities [7]. The platform excels at generating professional emails with proper tone, grammar, and punctuation, and assists with resume and cover letter creation tailored to specific industries [8].

### Educational and Learning Applications

Educational adoption has been substantial, with over 40% of college students using ChatGPT for schoolwork. Approximately 39% of potential students indicate they wouldn't consider attending a college that restricted ChatGPT and other AI tools [9]. The platform adapts its teaching style to individual learners' pace, offering personalized learning experiences with immediate feedback [8]. This educational integration represents a fundamental shift in how students approach learning and academic work.

### Coding and Development Use Cases

Software developers demonstrate the highest professional adoption rate at 79% [3]. ChatGPT assists with code generation, debugging, algorithm explanations, and programming guidance. The platform converts natural language into code snippets, provides debugging solutions, and aids in learning new programming languages [8]. Currently, 1.3 million U.S. developers have built projects on OpenAI's platform [3]. ChatGPT generates code examples and scripts for quick website fixes while supporting code completion scenarios [1].

### Business Process and Productivity Applications

ChatGPT provides substantial economic value through decision support, particularly important in knowledge-intensive jobs [6]. Using O*NET occupational data, researchers found that 58% of work-related messages involve two critical activities: obtaining and interpreting information, and decision-making/problem-solving [6].

Customer service applications show particular promise, with ChatGPT providing 24/7 support, handling multiple queries simultaneously, and learning from interactions to improve responses over time [8]. Tools like ChatGPT can increase customer service productivity by 30-45%, with support agents using AI handling 13.8% more customer inquiries per hour [4][6]. One compelling case study involved a Reddit user who landed three job interviews in under four days using ChatGPT-generated cover letters, after struggling to get one interview from 49 applications before using AI [1].

## Usage Patterns and Behavioral Trends

### Demographic Evolution

ChatGPT's user demographics have undergone significant evolution since launch. The gender gap has nearly closed, shifting from 80% male early adopters to 48% male/52% female by June 2025 [6]. Among users with names classifiable as either masculine or feminine, 37% had typically feminine names in January 2024, rising to more than half (52%) by July 2025 [2].

Age distribution shows 56.61% of users aged 18-34, with the largest single group being ages 25-34 (32.4%) [1][4]. More than 45% of users are under age 25, while the 30-44 age group shows the highest adoption rates at 17% usage [2]. Usage varies significantly by age cohort: 56% of 18-24 year-olds have used ChatGPT versus only 16% of those over 55 [3].

### Geographic Distribution and Global Reach

ChatGPT operates globally with availability in 161-188 countries and supports 59 languages [1][3]. The United States leads with 15-19% of ChatGPT users, followed by India (7.17%-8.71%), Brazil (5.05%-5.28%), Germany (4.04%), Canada (3.57%), and Japan (2.9%) [1][2][4]. These top five countries represent approximately 37% of the total user base [2].

Growth has been particularly strong in lower-income countries, and educated users in professional occupations are more likely to use ChatGPT for work [6]. However, the platform faces restrictions, being banned in 15 countries including China, Iran, Russia, North Korea, Cuba, and Syria [4][9].

### Professional Adoption Variations

Workplace usage varies dramatically across professions. 28% of employed U.S. adults use ChatGPT at work, up from 8% in 2023 [3]. Software developers lead professional adoption at 79%, while financial advisors show 34% total adoption with only 12% using it at work [3]. Other professional usage rates include 77% of marketers, 71% of consultants, and 67% of advertisers [4].

Workers estimate ChatGPT can reduce time on one-third of job tasks by half, with 80% of U.S. workers potentially having at least 10% of their tasks affected by GPTs [3]. This suggests widespread potential for productivity transformation across the economy.

### User Engagement and Session Metrics

User engagement metrics reveal sustained interaction patterns. Average session duration ranges from 7 minutes 48 seconds to 14 minutes 36 seconds depending on measurement methodology [3][4][9]. Users view 4.4-4.5 pages per visit with bounce rates between 30.94% and 40.01% [2][3][9].

The platform maintains strong daily engagement with over 1 billion messages sent daily and 18 billion messages weekly [3][6]. Over 81% of ChatGPT visits are direct traffic, with referrals accounting for 8.48% and organic search 7.68% [2]. Desktop usage dominates at 76.22% of visits, with 79.77% of traffic coming from direct visits [3]. The mobile app has achieved significant traction with over 500 million downloads on Google Play (4.5-star rating) and reaching 110 million installs with $30 million revenue in its first year [3][4].

## Productivity Impact and Economic Outcomes

### Individual User Productivity Gains

Rigorous academic research has quantified ChatGPT's productivity impact. A controlled MIT study by Shakked Noy and Whitney Zhang involving 444 experienced business professionals found dramatic improvements with ChatGPT use. Productivity increased by 59%: documents were completed in 17 minutes on average with ChatGPT versus 27 minutes without AI. Quality also improved significantly, with AI-assisted documents scoring 4.5 versus 3.8 on a 7-point scale [5].

The study revealed that time spent generating rough drafts was more than cut in half since most of this workload was offloaded to ChatGPT. Interestingly, time spent polishing drafts doubled, explaining both the speed and quality improvements [5]. This pattern suggests ChatGPT enables users to focus more time on refinement and strategic thinking rather than initial creation.

A Harvard/MIT study at Boston Consulting Group found consultants using GPT-4 finished tasks significantly faster than those without AI assistance [7]. These controlled studies provide robust evidence of measurable productivity gains across knowledge work.

### Business-Level Performance Improvements

Enterprise ChatGPT usage demonstrates substantial business-level improvements. The enterprise version shows a 40% increase in work quality [9]. Over 92% of Fortune 500 companies using the ChatGPT API report 75% faster code debugging [9]. Developers show remarkable productivity gains, with 126% more code output per week with AI assistance [6].

Marketing teams report 40-60% time savings in content creation [7]. Customer service sees 30-45% productivity increases, with agents handling 13.8% more inquiries per hour [4][6]. These improvements translate directly to operational efficiency and cost reduction.

### Financial Impact and Return on Investment

The financial impact of ChatGPT adoption is substantial. 25% of businesses using ChatGPT have saved $75,000 or more in their operations [4]. Among American companies, 25% report saving $50,000-$70,000, while 11% save over $100,000 [4]. These savings demonstrate measurable return on investment across organizations of various sizes.

ROI calculations vary by company size but consistently show strong returns. For startups (1-5 employees), monthly costs of €115 can generate €1,200-2,000 in savings with a 1-month break-even period and 900-1600% annual ROI. For SMEs (10-25 employees), costs of €230-575 can generate €3,000-6,000 in savings with 1-2 month break-even and 500-800% annual ROI [7].

However, implementation also brings workforce changes. 48% of companies using ChatGPT report it has replaced some workers, demonstrating both efficiency gains and workforce transformation [4]. Research suggests ChatGPT is more likely to complement human work rather than fully replace jobs, handling repetitive, data-driven tasks while allowing humans to focus on creative, strategic, or complex responsibilities [3].

### Industry-Specific Impact Analysis

Different industries show varying levels of productivity impact. In customer service, statistics indicate that a mere 5% increase in customer retention can lead to a 25% to 95% boost in profits [10]. Sales processes demonstrate 30% shorter sales cycles and 40% higher conversion rates with extensive sales automation [7].

Travel and hospitality lead in customer adoption at 18% with $1.48 trillion estimated economic impact [4]. The healthcare sector benefits from AI-powered imaging tools, patient management systems, and clinical decision support. Financial services leverage ChatGPT for risk management, fraud detection, and compliance automation [11]. Finance departments use ChatGPT for budget planning automation, forecasting with natural language input, real-time financial summaries, and executive-level dashboards [7].

## Global Trends and Market Position

### International Usage Patterns and Localization

ChatGPT's global reach extends across 161-188 countries with support for 59 languages, enabling worldwide communication and breaking down language barriers [3][8]. The platform becomes a pathway to connection, fostering better understanding and collaboration among people from diverse linguistic backgrounds [8].

Multilingual support enables real-time customer support across global teams, internal communications, and localization of product documentation and training materials [11]. Companies now build multilingual AI layers into communication tools like Slack, Microsoft Teams, and Zoom, bridging language barriers instantly [11]. This global accessibility has contributed significantly to ChatGPT's worldwide adoption.

### Competitive Landscape and Market Share

ChatGPT maintains dominant market position despite increasing competition from Claude, Google Gemini, Grok, DeepSeek, and Qwen [1]. The platform holds 82.7% overall market share, 62.5% of the AI assistant market, and 79.76% of the chatbot market [1][3][4]. This dominance persists even as competitors introduce new features and capabilities.

The competitive landscape continues evolving, but ChatGPT's first-mover advantage, extensive user base, and continuous improvement cycle maintain its market leadership. The platform's integration into business workflows and educational systems creates switching costs that help preserve market position.

### Future Growth Projections and Trends

Current trends suggest continued explosive growth. OpenAI projects reaching 1 billion ChatGPT users by the end of 2025 [2]. Revenue projections show growth from $3.7 billion in 2024 to $29.4 billion by 2026 [3]. These projections reflect both user base expansion and increasing monetization per user.

Usage patterns indicate ongoing evolution from primarily work-focused applications to broader personal and creative uses. The shift toward 70% non-work usage by June 2025 suggests ChatGPT is becoming integrated into daily life beyond professional contexts [6]. This trend toward ubiquitous adoption across all life domains indicates ChatGPT's transformation from a specialized tool to general-purpose technology infrastructure.

The closing gender gap, expansion into lower-income countries, and increasing adoption among older demographics all point toward continued mainstream adoption. As the technology becomes more accessible and user-friendly, adoption barriers continue falling across all segments of the global population.

### Sources

[1] Latest ChatGPT Statistics: 800M+ Users, Revenue (Oct 2025): https://nerdynav.com/chatgpt-statistics/
[2] Number of ChatGPT Users (October 2025): https://explodingtopics.com/blog/chatgpt-users
[3] 40+ ChatGPT Stats You Must Know in 2025: Usage, Growth & Impact: https://www.index.dev/blog/chatgpt-statistics
[4] ChatGPT Statistics in Companies [October 2025]: https://masterofcode.com/blog/chatgpt-statistics
[5] ChatGPT Revenue and Usage Statistics (2025): https://www.businessofapps.com/data/chatgpt-statistics/
[6] How People Use ChatGPT - OpenAI Research Paper: https://cdn.openai.com/pdf/a253471f-8260-40c6-a2cc-aa93fe9f142e/economic-research-chatgpt-usage-paper.pdf
[7] ChatGPT Statistics 2025: Usage, Growth, and Key Trends: https://thunderbit.com/blog/chatgpt-stats-usage-growth-trends
[8] What Can ChatGPT Do? 9 Compelling Use Cases in 2025: https://workhub.ai/chatgpt-use-cases/
[9] The Latest ChatGPT Statistics and User Trends (2022-2025): https://wisernotify.com/blog/chatgpt-users/
[10] 10 Top ChatGPT Use Cases in Marketing Industry: https://elandz.com/blog/martech/10-top-chatgpt-use-cases-in-marketing/
[11] Top 10 Use Cases for ChatGPT Enterprise Solutions in 2025: https://www.linkedin.com/pulse/top-10-use-cases-chatgpt-enterprise-solutions-2025-anu-geethan-eofje


⏱️ PARALLELISM EXPERIMENT WITH CLARIFICATION COMPLETED
⏱️ Total execution time: 577.8 seconds


## Parallelism Experiment Results

**Configuration Tested**: Increased from 1 to 5 concurrent researchers

**Results**:
- **Execution Time**: 577 seconds 
- **Report Quality**: More comprehensive with additional sources
- **System Issues**: Summarization timeouts after 120 seconds
- **Resource Cost**: 5x higher due to parallel processing

**Conclusion**: Higher parallelism improves research depth but increases computational overhead and may hit API limits. Optimal configuration likely 2-3 concurrent researchers.

## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs